# A8. Competitor Price Tracking & Dynamic Pricing Notebook

> **Related Module**: [A8 Pricing Strategy](../paths/a-operators/a8-pricing-strategy.md)
>
> **Features**: Competitor price change analysis + Price elasticity estimation + Dynamic pricing recommendations
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a8-price-tracker.ipynb)

In [ ]:
!pip install -q pandas numpy plotly

## 1. Input Price Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

np.random.seed(42)
days = 60
dates = pd.date_range(end=datetime.now(), periods=days, freq='D')

# Simulated price data (replace with your real data)
my_price = np.full(days, 39.99)
comp_a = 37.99 + np.random.normal(0, 0.5, days)
comp_a[40:] = 34.99 + np.random.normal(0, 0.3, days-40)  # Competitor A drops price on day 40
comp_b = np.full(days, 42.99) + np.random.normal(0, 0.3, days)
comp_c = 35.99 + np.random.normal(0, 0.4, days)
comp_c[50:] = 38.99 + np.random.normal(0, 0.3, days-50)  # Competitor C raises price on day 50

# Simulated sales (affected by competitor prices)
my_sales = 15 + np.random.poisson(3, days)
my_sales[40:] = my_sales[40:] - 3  # Sales drop after Competitor A's price cut

df = pd.DataFrame({
    'date': dates,
    'my_price': my_price,
    'comp_a_price': comp_a.round(2),
    'comp_b_price': comp_b.round(2),
    'comp_c_price': comp_c.round(2),
    'my_daily_sales': my_sales.clip(5)
})

print(f'Price tracking: {days} days')
print(f'My price: ${my_price[0]}')
print(f'Competitor A: ${comp_a[0]:.2f} → ${comp_a[-1]:.2f}')
print(f'Competitor B: ${comp_b[0]:.2f} → ${comp_b[-1]:.2f}')
print(f'Competitor C: ${comp_c[0]:.2f} → ${comp_c[-1]:.2f}')


## 2. Price Trend Visualization

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=df['my_price'], name='My Price', line=dict(width=3, color='blue')))
fig.add_trace(go.Scatter(x=df['date'], y=df['comp_a_price'], name='Competitor A', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=df['date'], y=df['comp_b_price'], name='Competitor B', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=df['date'], y=df['comp_c_price'], name='Competitor C', line=dict(dash='dash')))
fig.update_layout(title='60-Day Price Tracking', yaxis_title='Price ($)', xaxis_title='Date')
fig.show()

# Price gap analysis
print('=== Price Gap Analysis (Latest) ===')
for comp, col in [('Comp A', 'comp_a_price'), ('Comp B', 'comp_b_price'), ('Comp C', 'comp_c_price')]:
    gap = df['my_price'].iloc[-1] - df[col].iloc[-1]
    pct = gap / df['my_price'].iloc[-1] * 100
    emoji = '🔴' if gap > 3 else ('🟡' if gap > 0 else '🟢')
    print(f'{emoji} vs {comp}: ${gap:+.2f} ({pct:+.1f}%)')

## 3. Price Change Detection

In [ ]:
print('=== Price Change Alerts ===')
for comp, col in [('Competitor A', 'comp_a_price'), ('Competitor B', 'comp_b_price'), ('Competitor C', 'comp_c_price')]:
    prices = df[col]
    # Detect significant price changes (>5%)
    pct_changes = prices.pct_change()
    significant = df[abs(pct_changes) > 0.05]
    if len(significant) > 0:
        for _, row in significant.iterrows():
            change = pct_changes[row.name]
            direction = '📉 Price Drop' if change < 0 else '📈 Price Increase'
            print(f'{direction} {comp} on {row["date"].strftime("%m/%d")}: ${prices[row.name-1]:.2f} → ${row[col]:.2f} ({change*100:+.1f}%)')
    else:
        print(f'✅ {comp}: No significant changes')


## 4. Price-Sales Correlation

In [ ]:
# Analyze the impact of competitor price drops on our sales
df['price_gap_a'] = df['my_price'] - df['comp_a_price']
corr = df['price_gap_a'].corr(df['my_daily_sales'])
print(f'Price gap vs sales correlation: {corr:.3f}')
if corr < -0.3:
    print('⚠️ Strong negative correlation: when competitor A is cheaper, your sales drop')

fig = px.scatter(df, x='price_gap_a', y='my_daily_sales', trendline='ols',
                 title='Price Gap (Me - Comp A) vs My Daily Sales',
                 labels={'price_gap_a': 'Price Gap ($)', 'my_daily_sales': 'Daily Sales'})
fig.show()

# Pricing recommendations
print('\n=== Pricing Recommendations ===')
avg_gap = df['price_gap_a'].iloc[-7:].mean()
if avg_gap > 5:
    print(f'🔴 You are ${avg_gap:.2f} above Comp A. Consider:')
    print(f'   Option 1: Match at ${df["comp_a_price"].iloc[-1]:.2f} (risk: margin compression)')
    print(f'   Option 2: Partial match at ${(df["my_price"].iloc[-1] + df["comp_a_price"].iloc[-1])/2:.2f}')
    print(f'   Option 3: Hold price, differentiate on value (brand/quality/service)')
elif avg_gap > 0:
    print(f'🟡 Slight premium of ${avg_gap:.2f}. Monitor but no immediate action needed.')
else:
    print(f'🟢 You are competitively priced. Focus on conversion optimization.')


## 5. Export

In [ ]:
df.to_csv('price_tracking.csv', index=False)
print('Exported to price_tracking.csv')